# Assignment 6: Travelling Salesman Problem using Ant Colony Optimization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def make_cities(n):
    np.random.seed(42)
    cities = np.random.rand(n, 2) * 100
    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dist[i][j] = np.linalg.norm(cities[i] - cities[j])
    return cities, dist

def tour_length(tour, dist):
    n = len(tour)
    return sum(dist[tour[i]][tour[(i + 1) % n]] for i in range(n))

def run_aco(dist, n_ants, n_iter, alpha, beta, rho, Q=100):
    np.random.seed(0)
    n = len(dist)
    pheromone = np.ones((n, n))
    eta = 1 / (dist + np.eye(n))        # heuristic = 1/distance (eye avoids 1/0)
    best_tour, best_len, history = None, float('inf'), []

    for it in range(n_iter):
        tours = []
        for ant in range(n_ants):
            tour = [np.random.randint(n)]               # random start city
            while len(tour) < n:
                cur = tour[-1]
                prob = (pheromone[cur] ** alpha) * (eta[cur] ** beta)
                prob[tour] = 0                          # visited cities not allowed
                prob = prob / prob.sum()
                tour.append(np.random.choice(n, p=prob))
            tours.append(tour)

        pheromone *= (1 - rho)                          # evaporation

        for tour in tours:                              # deposit Q / L
            L = tour_length(tour, dist)
            for i in range(n):
                a, b = tour[i], tour[(i + 1) % n]
                pheromone[a][b] += Q / L
                pheromone[b][a] += Q / L
            if L < best_len:
                best_len, best_tour = L, tour

        history.append(best_len)
        print(f"Iteration {it + 1:2d}: best length = {best_len:.2f}")
    return best_tour, best_len, history

In [ ]:
n_cities, n_ants, n_iter = 10, 10, 50
alpha, beta, rho = 1.0, 2.0, 0.5
cities, dist = make_cities(n_cities)
best_tour, best_len, history = None, None, None

while True:
    print("\n========= ACO - TSP MENU =========")
    print(f"Cities = {n_cities}, Ants = {n_ants}, Iterations = {n_iter}, "
          f"alpha = {alpha}, beta = {beta}, rho = {rho}")
    print("1. Set number of cities")
    print("2. Set ACO parameters")
    print("3. Show distance matrix")
    print("4. Run ACO")
    print("5. Show best tour")
    print("6. Plot best tour and convergence")
    print("0. Exit")
    ch = input("Enter your choice: ").strip()
    try:
        if ch == '1':
            n_cities = int(input("Number of cities (3-50): "))
            if not 3 <= n_cities <= 50:
                print("Invalid. Using 10."); n_cities = 10
            cities, dist = make_cities(n_cities)
            best_tour = None
        elif ch == '2':
            n_ants = int(input("Number of ants: "))
            n_iter = int(input("Number of iterations: "))
            alpha  = float(input("Alpha (pheromone importance): "))
            beta   = float(input("Beta (distance importance): "))
            rho    = float(input("Evaporation rate rho (0-1): "))
        elif ch == '3':
            print(np.round(dist, 1))
        elif ch == '4':
            best_tour, best_len, history = run_aco(dist, n_ants, n_iter, alpha, beta, rho)
        elif ch == '5':
            if best_tour is None: print("Run ACO first (option 4)."); continue
            print("Best tour  :", [int(c) for c in best_tour] + [int(best_tour[0])])
            print(f"Best length: {best_len:.2f}")
        elif ch == '6':
            if best_tour is None: print("Run ACO first (option 4)."); continue
            route = list(best_tour) + [best_tour[0]]
            plt.figure(figsize=(10, 4))
            plt.subplot(1, 2, 1)
            plt.plot(cities[route, 0], cities[route, 1], 'o-')
            for i, (x, yy) in enumerate(cities): plt.text(x, yy, str(i))
            plt.title(f'Best Tour (length {best_len:.2f})')
            plt.subplot(1, 2, 2)
            plt.plot(history); plt.title('Convergence'); plt.xlabel('Iteration')
            plt.tight_layout(); plt.show()
        elif ch == '0':
            print("Exiting...")
            break
        else:
            print("Invalid choice")
    except ValueError:
        print("Please enter valid numbers")